In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

We will train a seq2seq model for machine translation. The dataset can be downloaded from the link - 
https://www.manythings.org/anki/

Example taken from https://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html

In [2]:
with open('deu-eng/deu.txt', encoding='utf-8') as fs:
    for line in fs:
        source_sent, target_sent, _ = line.strip().split("\t")
        print(f"{source_sent}||{target_sent}")
        break

Go.||Geh.


The files are all in Unicode, to simplify we will turn Unicode characters to ASCII, make everything lowercase, and trim most punctuation.

In [3]:
import unicodedata
import re

In [4]:
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

In [5]:
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s

In [6]:
all_pairs = []

with open('deu-eng/deu.txt', encoding='utf-8') as fs:
    for line in fs:
        pairs = line.strip().split("\t")[:-1]
        pairs = [normalizeString(s) for s in pairs]
        all_pairs.append(pairs)
        #print(pairs)

In [7]:
len(all_pairs)

261499

In [8]:
all_pairs[10]

['help !', 'zu hulf !']

In [9]:
# we will need this for both languages
word2index_src = {}
index2word_src = {}
word2index_tar = {}
index2word_tar = {}

src_i = 2 # 0 and 1 are special tokens
tar_i = 2 

for sent_pair in all_pairs:
    sent_src = sent_pair[0]
    sent_tar = sent_pair[1]
    
    for w in sent_src.split():
        if w not in word2index_src:
            word2index_src[w] = src_i
            index2word_src[src_i] = w
            src_i+=1
    
    for w in sent_tar.split():
        if w not in word2index_tar:
            word2index_tar[w] = tar_i
            index2word_tar[tar_i] = w
            tar_i+=1
    

In [10]:
len(word2index_src), len(word2index_tar)

(16538, 36708)

In [11]:
index2word_src[0] = '<SOS>'
index2word_tar[0] = '<SOS>'
index2word_src[1] = '<EOS>'
index2word_tar[1] = '<EOS>'
word2index_src['<SOS>'] = 0
word2index_tar['<SOS>'] = 0
word2index_src['<EOS>'] = 1
word2index_tar['<EOS>'] = 1

In [12]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input):
        embedded = self.embedding(input)
        output = embedded
        hidden = self.initHidden()
        all_outputs, last_hidden = self.gru(output, hidden)
        return last_hidden

    def initHidden(self):
        return torch.zeros( 1, self.hidden_size)

In [13]:
# We will decode one token at a time...
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden): # the first hidden state is obtained from the ecoder
        output = self.embedding(input)
        output = F.relu(output)
        all_output, last_hidden = self.gru(output, hidden)
        output = self.softmax(self.out(last_hidden))
        return output, last_hidden


In [14]:
# choose a random pair of sentence
sent_src, sent_tar = all_pairs[1000][0], all_pairs[1000][1]
print(sent_src)
print(sent_tar)

keep this .
behalte das !


In [15]:
sent_src_ind = [word2index_src[w] for w in sent_src.split()]
sent_tar_ind = [word2index_tar[w] for w in sent_tar.split()]

print(sent_src_ind)
print(sent_tar_ind)

[114, 161, 3]
[702, 112, 5]


In [16]:
# Add <SOS> and <EOS> token to each sentence..
# <SOS> = 0
# <EOS> = 1
sent_src_ind = [0] + sent_src_ind + [1]
sent_tar_ind = [0] + sent_tar_ind + [1]

print(sent_src_ind)
print(sent_tar_ind)

[0, 114, 161, 3, 1]
[0, 702, 112, 5, 1]


In [17]:
# Tensorize input 
src_tensor = torch.LongTensor(sent_src_ind)
tar_tensor = torch.LongTensor(sent_tar_ind)

In [18]:
# Training....
hidden_size = 512
input_size = len(word2index_src)
output_size = len(word2index_tar)
encoder = EncoderRNN(input_size=input_size, hidden_size=hidden_size)

enc_out = encoder(src_tensor)
#print(enc_out.shape)

decoder = DecoderRNN(output_size=output_size, hidden_size=hidden_size)

# We will train with teacher not enforcing...
# The first token will be the <SOS> token...
# We will use greedy decoding
encoder_optim = Adam(encoder.parameters(), lr=0.0001)
decoder_optim = Adam(decoder.parameters(), lr=0.0001)
criterion = nn.NLLLoss()

loss = 0
start = torch.LongTensor([0])
last_hidden = enc_out
for i in range(1, len(sent_tar_ind)-1):
    out, last_hidden = decoder(start, last_hidden)
    loss+= criterion(out, tar_tensor[i].reshape(1))
    start = torch.argmax(out).reshape(-1)

loss = loss/len(sent_tar_ind)
encoder_optim.zero_grad()
decoder_optim.zero_grad()

loss.backward()
encoder_optim.step()
decoder_optim.step()
    

In [19]:
# evaluation on a random sample
sent_src, sent_tar = all_pairs[100][0], all_pairs[100][1]
print(sent_src)
print(sent_tar)

really ?
wirklich ?


In [20]:
sent_src_ind = [word2index_src[w] for w in sent_src.split()]
sent_src_ind = [0] + sent_src_ind + [1]

In [21]:
sent_src_ind

[0, 63, 32, 1]

In [23]:
src_tensor = torch.LongTensor(sent_src_ind)
enc_out = encoder(src_tensor)

# Continue to generate until <EOS> token is generated or a particular length is reached

length = 5
start = torch.LongTensor([0])
gen_seq = [0]
h_n = enc_out
for _ in range(length):
    out, h_n = decoder(start, h_n)
    start = torch.argmax(out)
    w = start.item()
    gen_seq.append(w)
    if w==1:
        break
    start = start.reshape(-1)    
    

In [24]:
' '.join(index2word_tar[w] for w in gen_seq)

'<SOS> thron zahlungsarten weiterleiten unterdrucken aufzieht'

# Tasks

1) Split the translation data (i.e., parallel corpus) into training and test (80/20).  

2) Write a Pytorch dataset class for the translation task.

3) Write a train function to train the model on the translation task. Use ``teacher enforcing".